In [1]:
from pathlib import Path
import pandas as pd
import fastparquet

In [2]:
PROJECT_ROOT = Path.cwd().parent


In [17]:
dim_items = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/dim_items.parquet", engine="fastparquet")
item_genres = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/movies/item_genres.parquet", engine="fastparquet")

In [6]:
dim_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 45430 entries, 0 to 45429
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   item_id                45430 non-null  int64  
 1   source_item_id         45430 non-null  int64  
 2   title                  45430 non-null  string 
 3   description            44476 non-null  string 
 4   release_date           45346 non-null  string 
 5   runtime                45173 non-null  float64
 6   language               45430 non-null  string 
 7   popularity             45430 non-null  float64
 8   vote_average           45430 non-null  float64
 9   vote_count             45430 non-null  float64
 10  belongs_to_collection  4487 non-null   string 
 11  production_companies   45430 non-null  string 
 12  production_countries   45430 non-null  string 
 13  budget                 45430 non-null  int64  
 14  revenue                45430 non-null  float64
 15  poster_path  

Formula:

WR = R(v/v+m) + C(m/v+m)

Where:

R = movie average rating

v = vote count

C = mean rating across all movies

m = minimum votes threshold

In [7]:
C = dim_items["vote_average"].mean()

m = dim_items["vote_count"].quantile(0.90)

print(C)
print(m)

5.618329297820823
160.0


In [8]:
qualified = dim_items[
    dim_items["vote_count"] >= m
].copy()

In [9]:
qualified["weighted_rating"] = (
    (qualified["vote_count"] /
    (qualified["vote_count"] + m))
    * qualified["vote_average"]
    +
    (m /
    (qualified["vote_count"] + m))
    * C
)

In [10]:
top_movies = (
    qualified
    .sort_values(
        "weighted_rating",
        ascending=False
    )
    [["title",
      "vote_average",
      "vote_count",
      "weighted_rating"]]
    .head(20)
)

top_movies

,title,vote_average,vote_count,weighted_rating
314,The Shawshank Redemption,8.5,8358.0,8.445871
834,The Godfather,8.5,6024.0,8.425442
10306,Dilwale Dulhania Le Jayenge,9.1,661.0,8.421477
12477,The Dark Knight,8.3,12269.0,8.265479
2842,Fight Club,8.3,9678.0,8.256387
292,Pulp Fiction,8.3,8670.0,8.251408
522,Schindler's List,8.3,4436.0,8.206643
23655,Whiplash,8.3,4376.0,8.205408
5480,Spirited Away,8.3,3968.0,8.196059
2210,Life Is Beautiful,8.3,3643.0,8.187177


In [11]:
dim_items["vote_count"].describe(percentiles=[0.8, 0.9, 0.95, 0.99])

count    45430.000000
mean       109.935989
std        491.466335
min          0.000000
80%         50.000000
90%        160.000000
95%        434.000000
99%       2184.420000
max      14075.000000
Name: vote_count, dtype: float64

In [12]:
print("Qualified movies:", len(qualified))
print("Total movies:", len(dim_items))
print("Percentage:", len(qualified) / len(dim_items) * 100)

Qualified movies: 4551
Total movies: 45430
Percentage: 10.017609509134932


In [13]:
top_movies.info()

<class 'pandas.DataFrame'>
Index: 20 entries, 314 to 1170
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            20 non-null     string 
 1   vote_average     20 non-null     float64
 2   vote_count       20 non-null     float64
 3   weighted_rating  20 non-null     float64
dtypes: float64(3), string(1)
memory usage: 1.1 KB


In [16]:
top_movies.head(20)

,title,vote_average,vote_count,weighted_rating
314,The Shawshank Redemption,8.5,8358.0,8.445871
834,The Godfather,8.5,6024.0,8.425442
10306,Dilwale Dulhania Le Jayenge,9.1,661.0,8.421477
12477,The Dark Knight,8.3,12269.0,8.265479
2842,Fight Club,8.3,9678.0,8.256387
292,Pulp Fiction,8.3,8670.0,8.251408
522,Schindler's List,8.3,4436.0,8.206643
23655,Whiplash,8.3,4376.0,8.205408
5480,Spirited Away,8.3,3968.0,8.196059
2210,Life Is Beautiful,8.3,3643.0,8.187177


In [18]:
item_genres.info()

<class 'pandas.DataFrame'>
RangeIndex: 91006 entries, 0 to 91005
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   item_id  91006 non-null  int64 
 1   genre    91006 non-null  string
dtypes: int64(1), string(1)
memory usage: 2.0 MB


In [19]:
item_genres.head()

,item_id,genre
0,1,Animation
1,1,Comedy
2,1,Family
3,2,Adventure
4,2,Fantasy


In [20]:
def top_movies_by_genre(
    dim_items,
    item_genres,
    genre,
    percentile=0.90,
    top_n=20
):
    genre_movie_ids = item_genres.loc[
        item_genres["genre"] == genre,
        "item_id"
    ]

    movies = dim_items[
        dim_items["item_id"].isin(genre_movie_ids)
    ].copy()

    C = movies["vote_average"].mean()
    m = movies["vote_count"].quantile(percentile)

    qualified = movies[
        movies["vote_count"] >= m
    ].copy()

    qualified["weighted_rating"] = (
        (qualified["vote_count"] /
         (qualified["vote_count"] + m))
        * qualified["vote_average"]
        +
        (m /
         (qualified["vote_count"] + m))
        * C
    )

    return (
        qualified[
            ["title",
             "vote_average",
             "vote_count",
             "weighted_rating"]
        ]
        .sort_values(
            "weighted_rating",
            ascending=False
        )
        .head(top_n)
    )

In [21]:
sorted(item_genres["genre"].unique())

['Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Foreign',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western']

In [31]:
top_movies_by_genre(
    dim_items,
    item_genres,
    "Science Fiction"
)

,title,vote_average,vote_count,weighted_rating
15474,Inception,8.1,14075.0,7.982681
22863,Interstellar,8.1,11187.0,7.953998
1154,The Empire Strikes Back,8.2,5998.0,7.930175
256,Star Wars,8.1,6778.0,7.867055
1225,Back to the Future,8.0,6239.0,7.757883
23735,Guardians of the Galaxy,7.9,10014.0,7.749668
2457,The Matrix,7.9,9079.0,7.735184
1167,Return of the Jedi,7.9,4763.0,7.603182
1171,Alien,7.9,4564.0,7.591801
1163,A Clockwork Orange,8.0,3432.0,7.590401


In [30]:
item_genres.groupby("genre")["item_id"].nunique() \
    .sort_values(ascending=False)

genre
Drama              20243
Comedy             13176
Thriller            7618
Romance             6730
Action              6590
Horror              4670
Crime               4304
Documentary         3930
Adventure           3490
Science Fiction     3042
Family              2767
Mystery             2464
Fantasy             2309
Animation           1930
Foreign             1619
Music               1597
History             1398
War                 1322
Western             1042
TV Movie             765
Name: item_id, dtype: int64